In [8]:
import pm4py
import pandas as pd
from sklearn.model_selection import train_test_split

PREFIX_2013 = [10, 20, 30, 40, 50, 75, 100, 125, 150]
PREFIX_2017 = [100, 150, 200, 400, 800, 1000, 1200, 1400, 1500, 2000]
PREFIX_2018 = [100, 150, 200, 400, 800, 1000, 1200, 1400, 1500, 2000, 2500]
PREFIX_2019 = [100, 150, 200, 300, 400, 500, 600, 700, 800]

DATASETS = {
    "BPIC2013": ("data/BPI_Challenge_2013_incidents.xes", PREFIX_2013),
    "BPIC2017": ("data/BPI_Challenge_2017.xes", PREFIX_2017),
    "BPIC2018": ("data/BPI_Challenge_2018.xes", PREFIX_2018),
    "BPIC2019": ("data/BPI_Challenge_2019.xes", PREFIX_2019)
}


def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    event_log = pm4py.convert_to_dataframe(log)
    return event_log


def create_activity_sequences(df, prefix_length):
    sequences = []
    next_activities = []
    resources = []

    for resource, resource_df in df.groupby('org:resource'):
        activities = resource_df['concept:name'].values

        if len(activities) >= prefix_length + 1:
            prefix = activities[:prefix_length]
            next_activity = activities[prefix_length]

            sequences.append(prefix)
            next_activities.append(next_activity)
            resources.append(resource)

    sequences_df = pd.DataFrame(
        sequences,
        columns=[f"activity_{i+1}" for i in range(prefix_length)]
    )

    sequences_df['next_activity'] = next_activities
    sequences_df['org:resource'] = resources

    return sequences_df



def examine_class_distribution(sequences_df, dataset_name, prefix_length, split_name):
    counts = sequences_df['next_activity'].value_counts()

    summary = {
        "dataset": dataset_name,
        "prefix_length": prefix_length,
        "split": split_name,
        "number_of_samples": len(sequences_df),
        "number_of_resources": sequences_df['org:resource'].nunique(),
        "number_of_classes": len(counts),
        "classes_with_1_sample": (counts == 1).sum(),
        "classes_with_fewer_than_5_samples": (counts < 5).sum()
    }

    class_distribution = []

    for activity, count in counts.items():
        class_distribution.append({
            "dataset": dataset_name,
            "prefix_length": prefix_length,
            "split": split_name,
            "next_activity": activity,
            "count": count,
            "percentage": count / len(sequences_df) * 100
        })

    return summary, class_distribution



# Store the results
all_summaries = []
all_class_distributions = []

# Loop through all datasets
for dataset_name, (file_path, prefix_lengths) in DATASETS.items():

    print(f"\nLoading {dataset_name}...")

    event_log = import_xes(file_path)

    df = event_log[
        ['concept:name', 'org:resource', 'time:timestamp']
    ].copy()

    df = df.sort_values(
        by=['org:resource', 'time:timestamp']
    )

    # Loop through all prefix lengths for this dataset
    for prefix_length in prefix_lengths:

        print(f"Examining {dataset_name}, prefix length {prefix_length}")
        sequences_df = create_activity_sequences(df, prefix_length)
    
        # Split resources into training and test sets
        resources = sequences_df['org:resource'].unique()
    
        train_resources, test_resources = train_test_split(
            resources,
            test_size=0.2,
            random_state=42
        )
    
        train_df = sequences_df[
            sequences_df['org:resource'].isin(train_resources)
        ].copy()
    
        test_df = sequences_df[
            sequences_df['org:resource'].isin(test_resources)
        ].copy()
    
        # Examine the training set
        summary, class_distribution = examine_class_distribution(
            train_df, dataset_name, prefix_length, "train"
        )
    
        all_summaries.append(summary)
        all_class_distributions.extend(class_distribution)
    
        # Examine the test set
        summary, class_distribution = examine_class_distribution(
            test_df, dataset_name, prefix_length, "test"
        )
    
        all_summaries.append(summary)
        all_class_distributions.extend(class_distribution)


# Convert results into DataFrames
summary_df = pd.DataFrame(all_summaries)
class_distribution_df = pd.DataFrame(all_class_distributions)

# Save everything to an Excel file
with pd.ExcelWriter("notebooks/class_distribution_results.xlsx") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    class_distribution_df.to_excel(
        writer, sheet_name="Class distribution", index=False
    )

print("\nResults saved to class_distribution_results.xlsx")


Loading BPIC2013...


parsing log, completed traces :: 100%|██████████| 7554/7554 [00:03<00:00, 2276.22it/s]


Examining BPIC2013, prefix length 10
Examining BPIC2013, prefix length 20
Examining BPIC2013, prefix length 30
Examining BPIC2013, prefix length 40
Examining BPIC2013, prefix length 50
Examining BPIC2013, prefix length 75
Examining BPIC2013, prefix length 100
Examining BPIC2013, prefix length 125
Examining BPIC2013, prefix length 150

Loading BPIC2017...


parsing log, completed traces :: 100%|██████████| 31509/31509 [00:31<00:00, 1008.28it/s]


Examining BPIC2017, prefix length 100
Examining BPIC2017, prefix length 150
Examining BPIC2017, prefix length 200
Examining BPIC2017, prefix length 400
Examining BPIC2017, prefix length 800
Examining BPIC2017, prefix length 1000
Examining BPIC2017, prefix length 1200
Examining BPIC2017, prefix length 1400
Examining BPIC2017, prefix length 1500
Examining BPIC2017, prefix length 2000

Loading BPIC2018...


parsing log, completed traces :: 100%|██████████| 43809/43809 [02:09<00:00, 337.31it/s]


Examining BPIC2018, prefix length 100
Examining BPIC2018, prefix length 150
Examining BPIC2018, prefix length 200
Examining BPIC2018, prefix length 400
Examining BPIC2018, prefix length 800
Examining BPIC2018, prefix length 1000
Examining BPIC2018, prefix length 1200
Examining BPIC2018, prefix length 1400
Examining BPIC2018, prefix length 1500
Examining BPIC2018, prefix length 2000
Examining BPIC2018, prefix length 2500

Loading BPIC2019...


parsing log, completed traces :: 100%|██████████| 251734/251734 [00:46<00:00, 5431.30it/s]


Examining BPIC2019, prefix length 100
Examining BPIC2019, prefix length 150
Examining BPIC2019, prefix length 200
Examining BPIC2019, prefix length 300
Examining BPIC2019, prefix length 400
Examining BPIC2019, prefix length 500
Examining BPIC2019, prefix length 600
Examining BPIC2019, prefix length 700
Examining BPIC2019, prefix length 800

Results saved to class_distribution_results.xlsx
